# 폐암 CT 이미지 분류: 처음부터 학습하는 AlexNet

이 노트북은 폐 CT 이미지를 **Bengin, Malignant, Normal** 세 클래스로 분류합니다. `torchvision.models`의 사전학습 가중치나 YOLO는 사용하지 않고, 수업에서 구현한 AlexNet 구조를 직접 정의하여 처음부터 학습합니다.

> 의료 판단을 위한 도구가 아니라, 교육용 이미지 분류 실습입니다.

## 1. 라이브러리와 실행 환경

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms.functional import to_pil_image

In [ ]:
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print('사용 장치:', DEVICE)

## 2. 데이터 경로와 클래스별 이미지 수 확인

`ImageFolder`는 하위 폴더 이름을 클래스 이름으로 사용하고, 알파벳순으로 클래스 번호를 부여합니다.

In [ ]:
DATA_DIR = Path('data/The IQ-OTHNCCD lung cancer dataset')

if not DATA_DIR.exists():
    raise FileNotFoundError(f'데이터 폴더를 찾을 수 없습니다: {DATA_DIR}')

image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
class_dirs = sorted(path for path in DATA_DIR.iterdir() if path.is_dir())

for class_dir in class_dirs:
    image_count = sum(path.suffix.lower() in image_extensions for path in class_dir.iterdir())
    print(f'{class_dir.name}: {image_count}장')

## 3. 전처리와 학습·검증 데이터 분리

CT 이미지를 32×32으로 통일하고, 흑백 영상을 3채널로 복제하여 AlexNet의 입력 채널 수와 맞춥니다. 학습 데이터에만 작은 회전·이동을 적용하고, 검증 데이터에는 무작위 증강을 적용하지 않습니다.

원본 데이터에 별도 검증 폴더가 없으므로, 각 클래스 비율을 유지한 채 80%를 학습용, 20%를 검증용으로 나눕니다.

In [ ]:
IMAGE_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomAffine(degrees=8, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

In [ ]:
base_dataset = datasets.ImageFolder(DATA_DIR)
class_names = base_dataset.classes
targets = np.array(base_dataset.targets)
indices = np.arange(len(base_dataset))

train_indices, validation_indices = train_test_split(
    indices, test_size=0.2, random_state=SEED, stratify=targets
)

train_full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)
validation_full_dataset = datasets.ImageFolder(DATA_DIR, transform=eval_transform)
train_dataset = Subset(train_full_dataset, train_indices)
validation_dataset = Subset(validation_full_dataset, validation_indices)

print('클래스:', class_names)
print('클래스 번호:', base_dataset.class_to_idx)
print('학습 데이터:', len(train_dataset))
print('검증 데이터:', len(validation_dataset))

In [ ]:
BATCH_SIZE = 32
PIN_MEMORY = DEVICE.type == 'cuda'

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=PIN_MEMORY
)
validation_loader = DataLoader(
    validation_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=PIN_MEMORY
)

images, labels = next(iter(train_loader))
print('이미지 배치 shape:', images.shape)
print('레이블 배치 shape:', labels.shape)

## 4. AlexNet 모델 직접 구현

5개의 합성곱 계층과 3개의 완전 연결 계층으로 구성합니다. 마지막 출력 노드는 폐암 분류 클래스 수인 3개로 설정합니다. 사전학습 모델을 불러오지 않으므로 모든 가중치는 무작위 초기값에서 학습됩니다.

In [ ]:
class AlexNetLungCancer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(96, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((4, 4))
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)

model = AlexNetLungCancer(num_classes=len(class_names)).to(DEVICE)
print(model)

In [ ]:
dummy_input = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
with torch.inference_mode():
    logits = model(dummy_input)

print('모델 출력 shape:', logits.shape)
print('전체 파라미터 수:', sum(parameter.numel() for parameter in model.parameters()))

## 5. 손실 함수, 옵티마이저, 평가 지표

세 클래스 중 하나를 예측하는 다중 분류 문제이므로 `CrossEntropyLoss`를 사용합니다. 옵티마이저는 수업에서 사용한 `AdamW`이며, 평가 지표는 정확도(accuracy), 혼동행렬, 클래스별 precision·recall·F1-score입니다.

In [ ]:
EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

In [ ]:
def train_one_epoch(data_loader, model, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


def evaluate(data_loader, model, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.inference_mode():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples

## 6. 모델 학습

검증 손실이 가장 낮은 시점의 가중치만 저장합니다. 검증 손실이 연속 5회 개선되지 않으면 조기 종료합니다.

In [ ]:
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
BEST_MODEL_PATH = OUTPUT_DIR / 'alexnet_lung_cancer_best.pth'

history = {
    'train_loss': [], 'train_accuracy': [],
    'validation_loss': [], 'validation_accuracy': []
}
best_validation_loss = float('inf')
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = train_one_epoch(
        train_loader, model, criterion, optimizer, DEVICE
    )
    validation_loss, validation_accuracy = evaluate(
        validation_loader, model, criterion, DEVICE
    )
    scheduler.step(validation_loss)

    history['train_loss'].append(train_loss)
    history['train_accuracy'].append(train_accuracy)
    history['validation_loss'].append(validation_loss)
    history['validation_accuracy'].append(validation_accuracy)

    current_lr = optimizer.param_groups[0]['lr']
    print(
        f'Epoch [{epoch:02d}/{EPOCHS}] | lr={current_lr:.6f} | '
        f'train loss={train_loss:.4f}, acc={train_accuracy:.2%} | '
        f'val loss={validation_loss:.4f}, acc={validation_accuracy:.2%}'
    )

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f'최고 모델 저장: {BEST_MODEL_PATH}')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print('조기 종료: 검증 손실이 더 이상 개선되지 않았습니다.')
            break

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history['train_loss'], marker='o', label='Train loss')
plt.plot(epochs, history['validation_loss'], marker='o', label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history['train_accuracy'], marker='o', label='Train accuracy')
plt.plot(epochs, history['validation_accuracy'], marker='o', label='Validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid()
plt.show()

## 7. 검증 성능 평가와 해석

정확도만으로는 클래스별 성능 차이를 알기 어렵습니다. 특히 Malignant 이미지가 가장 많으므로, 혼동행렬과 클래스별 precision·recall·F1-score를 함께 확인합니다. Malignant의 recall이 낮다면 실제 악성 이미지를 놓치는 경우가 많다는 뜻이므로 의료 목적에서는 특히 주의가 필요합니다.

In [ ]:
try:
    best_state_dict = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True)
except TypeError:
    best_state_dict = torch.load(BEST_MODEL_PATH, map_location=DEVICE)

model.load_state_dict(best_state_dict)
validation_loss, validation_accuracy = evaluate(
    validation_loader, model, criterion, DEVICE
)
print(f'최고 검증 손실: {validation_loss:.4f}')
print(f'검증 정확도: {validation_accuracy:.2%}')

In [ ]:
def collect_predictions(data_loader, model, device):
    model.eval()
    true_labels = []
    predicted_labels = []

    with torch.inference_mode():
        for images, labels in data_loader:
            logits = model(images.to(device))
            predictions = logits.argmax(dim=1).cpu()
            true_labels.extend(labels.tolist())
            predicted_labels.extend(predictions.tolist())

    return np.array(true_labels), np.array(predicted_labels)

true_labels, predicted_labels = collect_predictions(validation_loader, model, DEVICE)
matrix = confusion_matrix(true_labels, predicted_labels)
print('혼동행렬')
print(matrix)
print()
print(classification_report(
    true_labels, predicted_labels, target_names=class_names, digits=4, zero_division=0
))

In [ ]:
plt.figure(figsize=(7, 5))
plt.imshow(matrix, interpolation='nearest', cmap='Blues')
plt.title('Validation Confusion Matrix')
plt.colorbar()
ticks = np.arange(len(class_names))
plt.xticks(ticks, class_names, rotation=25)
plt.yticks(ticks, class_names)

threshold = matrix.max() / 2
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        value = matrix[row, column]
        plt.text(
            column, row, str(value), ha='center', va='center',
            color='white' if value > threshold else 'black'
        )

plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()

## 8. 검증 이미지 6장 예측

각 이미지에 대해 실제 클래스, 예측 클래스, 가장 높은 예측 확률을 표시합니다. 예측 클래스와 실제 클래스가 다르면 오분류 사례이며, 영상 품질·병변 형태·클래스 불균형 등을 추가로 확인해야 합니다.

In [ ]:
def denormalize(image):
    mean = torch.tensor((0.5, 0.5, 0.5), dtype=image.dtype).view(3, 1, 1)
    std = torch.tensor((0.5, 0.5, 0.5), dtype=image.dtype).view(3, 1, 1)
    return (image.cpu() * std + mean).clamp(0, 1)

sample_indices = random.sample(
    range(len(validation_dataset)), k=min(6, len(validation_dataset))
)

model.eval()
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

with torch.inference_mode():
    for axis, sample_index in zip(axes.flatten(), sample_indices):
        image, label = validation_dataset[sample_index]
        logits = model(image.unsqueeze(0).to(DEVICE))
        probabilities = torch.softmax(logits, dim=1).squeeze(0).cpu()
        prediction = probabilities.argmax().item()

        axis.imshow(to_pil_image(denormalize(image)))
        axis.set_title(
            f'실제: {class_names[label]}\n'
            f'예측: {class_names[prediction]} ({probabilities[prediction]:.1%})'
        )
        axis.axis('off')

plt.tight_layout()
plt.show()

## 9. 의료 현장 적용 시 고려할 문제점

1. **오진 위험과 책임**: 검증 정확도가 높아도 실제 환자·촬영 장비·병원 환경이 달라지면 성능이 달라질 수 있습니다. 특히 악성 사례를 정상 또는 양성으로 예측하는 false negative는 치료 지연으로 이어질 수 있으므로, 모델 결과는 의사의 판독을 보조하는 용도로만 사용하고 최종 판단을 대신해서는 안 됩니다.

2. **데이터 편향과 개인정보 보호**: 이 데이터셋의 클래스 수는 균형적이지 않고, 특정 기관·장비·인구집단의 영상에 치우쳐 있을 수 있습니다. 이러한 모델은 다른 병원이나 환자 집단에서 불공정하거나 낮은 성능을 보일 수 있습니다. 실제 도입 때는 외부 데이터 검증, 지속적인 성능 감시, 영상·환자정보의 비식별화와 안전한 접근 통제가 필요합니다.